In [1]:
# from waybackpy import WaybackMachineCDXServerAPI
 
# CDN_HOSTS = [
#     "cdn.akamai.steamstatic.com",
#     "cdn.fastly.steamstatic.com",
#     "cdn.cloudflare.steamstatic.com",
#     "steamcdn-a.akamaihd.net",
#     "cdn.steamstatic.com",
#     "cdn2.steamstatic.com",
#     "media.steampowered.com",
# ]
 
# KNOWN_PATH = "/steam/publicstats/contentserver_bandwidth_stacked.jsonp"
 
# seen = set()
# urls = []
 
# for host in CDN_HOSTS:
#     url = f"https://{host}{KNOWN_PATH}"
#     print(f"Querying {host}...", flush=True)
#     try:
#         cdx = WaybackMachineCDXServerAPI(url, match_type="prefix")
#         for snapshot in cdx.snapshots():
#             if snapshot.archive_url not in seen:
#                 seen.add(snapshot.archive_url)
#                 urls.append(snapshot.archive_url)
#     except Exception as e:
#         print(f"  ERROR: {e}")
 
# with open("wayback_urls_all.txt", "w") as f:
#     for url in sorted(urls):
#         f.write(url + "\n")
 
# print(f"\nDone. {len(urls)} URLs written to wayback_urls_all.txt")

In [2]:
with open("jsonp_urls.txt") as f:
    old_urls = set(line.strip() for line in f if line.strip())

In [3]:
with open("wayback_urls_all.txt") as f:
    new_urls = set(line.strip() for line in f if line.strip())

In [4]:
diff = new_urls - old_urls
 
print(f"Old set: {len(old_urls)} URLs")
print(f"New set: {len(new_urls)} URLs")
print(f"New URLs not in old set: {len(diff)}\n")

Old set: 729 URLs
New set: 904 URLs
New URLs not in old set: 888



In [5]:
diff_list = sorted(diff)

In [6]:
import requests
import time
import pandas as pd
from json import loads
from functools import reduce
from fake_useragent import UserAgent
from tqdm import tqdm

SCRAPE_JSONP = True

In [7]:
if SCRAPE_JSONP:
    all_dfs = []

    bad_urls = []
    for url in tqdm(diff_list):
        try:
            ua = UserAgent()
            headers = {"User-Agent": ua.random}
            response = requests.get(url, headers=headers, timeout=300)
        except requests.RequestException as e:
            print(f"Error fetching URL {url}: {e}")
            bad_urls.append(url)
            continue
        startidx = response.text.find("(")
        endidx = response.text.find(")")

        try:
            data = loads(response.text[startidx + 1 : endidx])
        except Exception as e:
            print(f"Error parsing JSONP from URL {url}: {e}")
            continue

        if "json" not in data:
            time.sleep(10)
            continue

        series_list = loads(data["json"])

        df_list = []
        for series in series_list:
            df_dict = {}
            region = series["label"]
            df_dict["Timestamp"] = [
                pd.to_datetime(x[0], unit="ms") for x in series["data"]
            ]
            df_dict[region] = [int(x[1]) for x in series["data"]]
            df_list.append(pd.DataFrame(df_dict))

        df = reduce(
            lambda x, y: pd.merge(x, y, on="Timestamp", how="outer"), df_list
        )
        df = df.sort_values("Timestamp").reset_index(drop=True)
        all_dfs.append(df)
        time.sleep(10)
    
    for url in bad_urls:
        try:
            ua = UserAgent()
            headers = {"User-Agent": ua.random}
            response = requests.get(url, headers=headers, timeout=300)
        except requests.RequestException as e:
            print(f"Error fetching URL {url}: {e}")
            continue
        startidx = response.text.find("(")
        endidx = response.text.find(")")

        try:
            data = loads(response.text[startidx + 1 : endidx])
        except Exception as e:
            print(f"Error parsing JSONP from URL {url}: {e}")
            continue

        if "json" not in data:
            time.sleep(10)
            continue

        series_list = loads(data["json"])

        df_list = []
        for series in series_list:
            df_dict = {}
            region = series["label"]
            df_dict["Timestamp"] = [
                pd.to_datetime(x[0], unit="ms") for x in series["data"]
            ]
            df_dict[region] = [int(x[1]) for x in series["data"]]
            df_list.append(pd.DataFrame(df_dict))

        df = reduce(
            lambda x, y: pd.merge(x, y, on="Timestamp", how="outer"), df_list
        )
        df = df.sort_values("Timestamp").reset_index(drop=True)
        all_dfs.append(df)
        time.sleep(10)

    df_combined = pd.concat(all_dfs, axis=0, ignore_index=True)
    df_combined.to_csv("old_data.csv", index=False)

  0%|          | 0/888 [00:00<?, ?it/s]

  0%|          | 1/888 [00:21<5:12:09, 21.12s/it]

Error fetching URL https://web.archive.org/web/20141229111629/http://cdn.akamai.steamstatic.com:80/steam/publicstats/contentserver_bandwidth_stacked.jsonp?v=12-24-2014-19: HTTPSConnectionPool(host='web.archive.org', port=443): Max retries exceeded with url: /web/20141229111629/http://cdn.akamai.steamstatic.com:80/steam/publicstats/contentserver_bandwidth_stacked.jsonp?v=12-24-2014-19 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x0000027CAB7667B0>, 'Connection to web.archive.org timed out. (connect timeout=300)'))


  0%|          | 2/888 [00:42<5:12:04, 21.13s/it]

Error fetching URL https://web.archive.org/web/20150107014909/http://cdn.akamai.steamstatic.com:80/steam/publicstats/contentserver_bandwidth_stacked.jsonp?v=12-31-2014-05: HTTPSConnectionPool(host='web.archive.org', port=443): Max retries exceeded with url: /web/20150107014909/http://cdn.akamai.steamstatic.com:80/steam/publicstats/contentserver_bandwidth_stacked.jsonp?v=12-31-2014-05 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x0000027CAB800E10>, 'Connection to web.archive.org timed out. (connect timeout=300)'))


...

Error fetching URL https://web.archive.org/web/20230712163841/https://cdn.akamai.steamstatic.com/steam/publicstats/contentserver_bandwidth_stacked.jsonp?v=03-29-2023-21: HTTPSConnectionPool(host='web.archive.org', port=443): Max retries exceeded with url: /web/20230712163841/https://cdn.akamai.steamstatic.com/steam/publicstats/contentserver_bandwidth_stacked.jsonp?v=03-29-2023-21 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x0000027C897F8050>, 'Connection to web.archive.org timed out. (connect timeout=300)'))


 87%|████████▋ | 776/888 [3:49:06<27:11, 14.57s/it]  

Error fetching URL https://web.archive.org/web/20251201054230/https://cdn.fastly.steamstatic.com/steam/publicstats/contentserver_bandwidth_stacked.jsonp?v=12-01-2025-05: HTTPSConnectionPool(host='web.archive.org', port=443): Max retries exceeded with url: /web/20251201054230/https://cdn.fastly.steamstatic.com/steam/publicstats/contentserver_bandwidth_stacked.jsonp?v=12-01-2025-05 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x0000027C897F8050>, 'Connection to web.archive.org timed out. (connect timeout=300)'))


100%|██████████| 888/888 [4:10:32<00:00, 16.93s/it]


Error fetching URL https://web.archive.org/web/20220119140703/https://cdn.cloudflare.steamstatic.com/steam/publicstats/contentserver_bandwidth_stacked.jsonp?v=01-19-2022-14: HTTPSConnectionPool(host='web.archive.org', port=443): Max retries exceeded with url: /web/20220119140703/https://cdn.cloudflare.steamstatic.com/steam/publicstats/contentserver_bandwidth_stacked.jsonp?v=01-19-2022-14 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x0000027CAB673ED0>, 'Connection to web.archive.org timed out. (connect timeout=300)'))


In [8]:
current_df = pd.read_csv("../data/bandwidths.csv", parse_dates=["Timestamp"])
current_df

,Timestamp,Central America,Africa,Middle East,Oceania,South America,Russia,Asia,Europe,North America
0,2016-10-03 10:20:00,2,5.0,29.0,55.0,31.0,153.0,386.0,465.0,142.0
1,2016-10-03 10:30:00,2,5.0,30.0,53.0,31.0,154.0,391.0,473.0,139.0
2,2016-10-03 10:40:00,2,5.0,31.0,53.0,32.0,154.0,392.0,484.0,134.0
3,2016-10-03 10:50:00,2,5.0,31.0,51.0,32.0,158.0,390.0,498.0,134.0
4,2016-10-03 11:00:00,2,5.0,32.0,51.0,33.0,167.0,402.0,511.0,133.0
...,...,...,...,...,...,...,...,...,...,...
155686,2026-03-13 02:10:00,194,89.0,920.0,495.0,3473.0,607.0,5607.0,2153.0,11721.0
155687,2026-03-13 02:20:00,194,89.0,903.0,497.0,3298.0,607.0,5928.0,2058.0,11555.0
155688,2026-03-13 02:30:00,196,88.0,882.0,522.0,3206.0,594.0,6214.0,1992.0,11491.0
155689,2026-03-13 02:40:00,196,85.0,870.0,522.0,3116.0,594.0,6426.0,1928.0,11338.0


In [9]:
old_df = (
    pd.read_csv("old_data.csv", parse_dates=["Timestamp"])
    .sort_values("Timestamp")
    .reset_index(drop=True)
)
old_df = old_df.loc[
    (old_df["Timestamp"].dt.minute % 10 == 0)
    & (old_df["Timestamp"].dt.second == 0)
]
old_df["nan_count"] = old_df.isnull().sum(axis=1)
old_df = (
    old_df
    .sort_values(by=["Timestamp", "nan_count"])
    .drop_duplicates(subset="Timestamp", keep="first")
    .reset_index(drop=True)
)
old_df = old_df.drop(columns=["nan_count"])

# Filter to values not in current_df
old_df = old_df.loc[~old_df["Timestamp"].isin(current_df["Timestamp"])]
old_df

,Timestamp,Central America,Africa,Middle East,Oceania,South America,Russia,Asia,Europe,North America
109,2019-06-15 03:50:00,10.0,10.0,50.0,146.0,258.0,124.0,1435,315,1373.0
110,2019-06-15 04:00:00,11.0,10.0,49.0,148.0,249.0,125.0,1489,306,1348.0
111,2019-06-15 04:10:00,11.0,10.0,49.0,151.0,244.0,127.0,1532,299,1322.0
112,2019-06-15 04:20:00,11.0,10.0,48.0,148.0,239.0,128.0,1560,299,1309.0
113,2019-06-15 04:30:00,10.0,10.0,48.0,148.0,234.0,134.0,1608,296,1265.0
...,...,...,...,...,...,...,...,...,...,...
36960,2024-05-14 10:10:00,18.0,56.0,414.0,458.0,286.0,1203.0,9422,2999,1499.0
36961,2024-05-14 10:20:00,19.0,59.0,477.0,468.0,322.0,1329.0,9995,3629,1499.0
36962,2024-05-14 10:30:00,19.0,61.0,526.0,471.0,331.0,1389.0,10347,3906,1469.0
36963,2024-05-14 10:40:00,19.0,62.0,559.0,471.0,331.0,1453.0,10877,4117,1395.0


In [10]:
combined = pd.concat([old_df, current_df], axis=0, ignore_index=True).sort_values("Timestamp").reset_index(drop=True)
regions_all = [
    "Central America",
    "Africa",
    "Middle East",
    "Oceania",
    "South America",
    "Russia",
    "Asia",
    "Europe",
    "North America",
]
combined[regions_all] = combined[regions_all].astype("Int64")
combined.to_csv("../data/bandwidths.csv", index=False)